# Long, then wide again

Melting the week produces 20 rows. Pivoting those rows back, with the reader as the index and the day as the columns, restores each cell. The grand total is 89 again. Pivot's column order follows the sorted day labels, so the columns come back as Fri, Mon, Thu, Tue, Wed. Ada's Monday is still 4. The values did not follow the column order off the page.

A second tiny table records Ada's Monday twice, as 4 and as 1. `pivot` refuses it: one reader and one day would have to become one cell, and two numbers cannot occupy that cell. `pivot_table` with `sum` is allowed to combine them:

$$
4 + 1 = 5.
$$


In [1]:
# The reading desk for the week of 5 October 2026. Monday is the morning note.
import pandas as pd

def week_frame():
    return pd.DataFrame(
        {
            "Mon": [4, 6, 3, 5],
            "Tue": [5, 6, 0, 7],
            "Wed": [3, 6, 8, 2],
            "Thu": [6, 6, 1, 4],
            "Fri": [2, 6, 4, 5],
        },
        index=["Ada", "Grace", "Alan", "Linus"],
    )
# Round-trip the week. A duplicate Monday has to be summed on purpose.
week = week_frame()
long = week.reset_index(names="name").melt(
    id_vars="name", var_name="day", value_name="pages"
)
wide = long.pivot(index="name", columns="day", values="pages")
print(list(wide.columns), int(wide.sum().sum()), int(wide.loc["Ada", "Mon"]))
assert int(wide.loc["Ada", "Mon"]) == 4
assert int(wide.sum().sum()) == 89
assert list(wide.columns) == ["Fri", "Mon", "Thu", "Tue", "Wed"]
tiny = pd.DataFrame(
    {"name": ["Ada", "Ada"], "day": ["Mon", "Mon"], "pages": [4, 1]}
)
try:
    tiny.pivot(index="name", columns="day", values="pages")
    failed = False
except ValueError:
    failed = True
assert failed
combined = tiny.pivot_table(index="name", columns="day", values="pages", aggfunc="sum")
assert int(combined.loc["Ada", "Mon"]) == 5


['Fri', 'Mon', 'Thu', 'Tue', 'Wed'] 89 4


## Pitfall

`pivot` is a reshape. It keeps a number only when each pair of labels occurs once. `pivot_table` is an aggregation. Using it on the clean week also works, and with `sum` it returns the same 89, because each pair occurs once. On Ada's two Mondays the two functions part: one refuses, the other reports 5.

## Summary

The melted week pivots back to 89 pages, with Ada's Monday equal to 4, and with the columns sorted by day name. Two Mondays of 4 and 1 become 5 only when the reshape is told to add.
